# Ujian Tengah Semester - Deep Learning
## Klasifikasi Judul Berita Clickbait Berbahasa Indonesia Menggunakan TF-IDF dan Multilayer Perceptron (MLP)

> **Kelompok  2**

**Ketua/Koordinator:** Ary Sigit Julian (NIM 2330105030001)

**Anggota Kelompok:**
- Febrian Nur Hidayat (NIM 2330105030026)
- Naufal Fakhri (NIM 2330105030011)
- Raydamar Rizkyaka Riyadi (NIM 2330205030035)
- Rinaldi Rizwar (NIM 2330205030036)

---

## Deskripsi Proyek

Proyek Ujian Tengah Semester (UTS) ini bertujuan untuk membangun model klasifikasi judul berita berbahasa Indonesia ke dalam dua kategori, yaitu *clickbait* dan *non-clickbait*. Dataset yang digunakan adalah CLICK-ID, dengan data yang dipilih berdasarkan kesepakatan ketiga anotator (*All-Agree*).

Proses pengembangan mencakup pemeriksaan dan pembersihan dataset (*preprocessing*), pembagian data, ekstraksi fitur menggunakan *Term Frequency-Inverse Document Frequency* (TF-IDF), serta pelatihan model *Multilayer Perceptron* (MLP). Sebagai pembanding, digunakan model Logistic Regression dan Naive Bayes.

Evaluasi dilakukan menggunakan metrik klasifikasi, kurva *training* dan *validation*, serta *confusion matrix* untuk menganalisis performa model.


# Bagian Awal - Persiapan *Environment*

## 1 - Import Library

Mengimpor seluruh library yang digunakan untuk pengolahan dataset, preprocessing teks, ekstraksi fitur TF-IDF, pelatihan model, evaluasi, visualisasi, dan penyimpanan hasil.

In [ ]:
# ============================================
# 1. IMPORT LIBRARY
# ============================================

# Operasi sistem, file, dan pengolahan teks
import os
import re
import shutil
import subprocess
from io import BytesIO

# Pengolahan data dan numerik
import pandas as pd
import numpy as np
import scipy.sparse as sp
import joblib

# Visualisasi
import matplotlib.pyplot as plt
from IPython.display import display

# Integrasi Google Colab
from google.colab import files

# Pembagian data dan ekstraksi fitur
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Model baseline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

# Pembobotan kelas
from sklearn.utils.class_weight import compute_class_weight

# Evaluasi model
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

# Deep Learning - TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

print("[OK] Seluruh library berhasil diimpor.")
print("TensorFlow Version:", tf.__version__)

## 2 - Persiapan Direktori Runtime

Menyiapkan direktori penyimpanan sementara pada lingkungan Google Colab untuk menampung dataset, model terlatih, hasil evaluasi, dan visualisasi. Seluruh proses dijalankan tanpa memerlukan koneksi Google Drive.

In [ ]:
# ============================================
# 2. PERSIAPAN DIREKTORI RUNTIME
# ============================================

# Direktori utama
BASE_DIR = "/content"

# Direktori dataset
DATA_DIR = os.path.join(BASE_DIR, "data")

# Direktori hasil training dan evaluasi
OUTPUT_DIR = os.path.join(BASE_DIR, "output_model")

# Membuat folder jika belum tersedia
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Path dataset utama
DATA_PATH = os.path.join(DATA_DIR, "main_dataset.xlsx")

print("[OK] Direktori runtime berhasil disiapkan.")
print("Dataset :", DATA_DIR)
print("Output  :", OUTPUT_DIR)

# Bagian 1 - *Preprocessing* & *Cleaning* Dataset

## Step 1: Mengimpor Dataset

Mengambil dataset CLICK-ID dari repositori GitHub secara otomatis. Apabila pengunduhan gagal atau file tidak valid, dataset dapat diunggah secara manual melalui Google Colab.

In [ ]:
# ============================================
# STEP 1: MENGIMPOR DATASET
# ============================================

# Isi dengan URL RAW file dari GitHub
DATA_URL = "https://github.com/nfakhrii/kelompok2-deeplearning/raw/refs/heads/main/data/main_dataset.xlsx"

# Menggunakan DATA_PATH dari Bagian Awal
# /content/data/main_dataset.xlsx


def validasi_excel(path):
    try:
        with pd.ExcelFile(path) as excel:
            return "All-Agree" in excel.sheet_names
    except Exception:
        return False


# Hapus file lama agar tidak tertukar
if os.path.exists(DATA_PATH):
    os.remove(DATA_PATH)

berhasil = False

# Opsi 1: Download otomatis melalui GitHub
if DATA_URL.strip():
    print("Mengunduh dataset dari GitHub...")

    try:
        hasil = subprocess.run(
            [
                "wget",
                "--quiet",
                "--timeout=30",
                "--tries=2",
                "-O", DATA_PATH,
                DATA_URL
            ],
            capture_output=True,
            text=True,
            timeout=90
        )

        berhasil = (
            hasil.returncode == 0
            and validasi_excel(DATA_PATH)
        )

    except Exception as e:
        print("Pengunduhan gagal:", e)

    if berhasil:
        print("[OK] Dataset berhasil diunduh dari GitHub.")
    else:
        print("[INFO] Download gagal atau file tidak valid.")


# Opsi 2: Upload manual
if not berhasil:
    print("\nSilakan upload main_dataset.xlsx")

    uploaded = files.upload()

    if len(uploaded) != 1:
        raise ValueError(
            "Upload tepat satu file main_dataset.xlsx."
        )

    nama_file, isi_file = next(iter(uploaded.items()))

    with open(DATA_PATH, "wb") as f:
        f.write(isi_file)

    if not validasi_excel(DATA_PATH):
        raise ValueError(
            "File Excel tidak valid atau sheet "
            "'All-Agree' tidak ditemukan."
        )

    print("[OK] Dataset berhasil diunggah manual.")


print("\nLokasi dataset:", DATA_PATH)
print("Ukuran file:", round(
    os.path.getsize(DATA_PATH) / (1024 * 1024), 2
), "MB")

## Step 2: Membaca Dataset All-Agree

Membaca sheet All-Agree yang berisi judul berita dengan kesepakatan label dari tiga anotator. Data yang digunakan terdiri atas kolom judul berita dan label klasifikasi, kemudian label diubah menjadi representasi numerik untuk pelatihan model.

In [ ]:
# ============================================
# STEP 2: MEMBACA DATASET ALL-AGREE
# ============================================

raw = pd.read_excel(
    DATA_PATH,
    sheet_name="All-Agree",
    header=None
)

# Ambil data mulai baris ketiga
# Kolom 0 = judul, kolom 7 = label akhir
df = raw.iloc[2:, [0, 7]].copy()

# Simpan nomor baris Excel untuk keperluan audit
df["excel_row"] = df.index + 1

df.columns = ["title", "label", "excel_row"]

# Normalisasi format data
df["title"] = (
    df["title"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df["label"] = (
    df["label"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Ambil hanya data yang valid
df = df[
    df["title"].ne("")
    & df["label"].isin([
        "clickbait", "non-clickbait"
    ])
].copy()

# Encoding label
df["label_score"] = df["label"].map({
    "non-clickbait": 0,
    "clickbait": 1
}).astype(int)

df = df[
    ["excel_row", "title", "label", "label_score"]
].reset_index(drop=True)

print("=== DATASET ALL-AGREE ===")
print("Jumlah data:", len(df))

print("\nDistribusi label:")
print(df["label"].value_counts())

display(df.head(10))

assert len(df) == 8613, (
    f"Jumlah data tidak sesuai: {len(df)}"
)

## Step 3: Pemeriksaan Kualitas Data Awal

Memeriksa judul berita yang memiliki label bertentangan serta judul duplikat sebelum proses cleaning. Seluruh baris dengan label bertentangan dikeluarkan, sedangkan duplikat berlabel konsisten disaring dengan mempertahankan kemunculan pertama.

In [ ]:
# ============================================
# STEP 3: PEMERIKSAAN KUALITAS DATA AWAL
# ============================================

jumlah_awal = len(df)

# Kunci sementara untuk mendeteksi judul identik
# tanpa memperhatikan kapitalisasi dan spasi
df["title_key"] = (
    df["title"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# 1. Identifikasi konflik label
label_per_judul = (
    df.groupby("title_key")["label"].nunique()
)

judul_konflik = label_per_judul[
    label_per_judul > 1
].index

audit_konflik = df[
    df["title_key"].isin(judul_konflik)
].copy()

print("=== AUDIT KONFLIK LABEL ===")
display(
    audit_konflik[
        ["excel_row", "title", "label"]
    ]
)

# Hapus semua baris dengan konflik label
df = df[
    ~df["title_key"].isin(judul_konflik)
].copy()

jumlah_konflik = len(audit_konflik)

# 2. Identifikasi duplikat awal
audit_duplikat = df[
    df.duplicated("title_key", keep=False)
].copy()

print("\n=== AUDIT DUPLIKAT AWAL ===")
display(
    audit_duplikat[
        ["excel_row", "title", "label"]
    ].sort_values("title")
)

# Pertahankan kemunculan pertama
sebelum_dedup = len(df)

df = df.drop_duplicates(
    subset="title_key",
    keep="first"
).copy()

jumlah_duplikat = sebelum_dedup - len(df)

# Hapus kolom sementara
df = df.drop(columns=["title_key"])
df = df.reset_index(drop=True)

print("\n=== HASIL PEMERIKSAAN DATA ===")
print(f"Jumlah data awal       : {jumlah_awal}")
print(f"Baris konflik dihapus  : {jumlah_konflik}")
print(f"Duplikat awal dihapus  : {jumlah_duplikat}")
print(f"Jumlah data tersisa    : {len(df)}")

assert jumlah_konflik == 3
assert jumlah_duplikat == 12
assert len(df) == 8598

print("\n[OK] Pemeriksaan data awal selesai.")

## Step 4: Text Preprocessing dan Cleaning

Melakukan dua tahap preprocessing teks, yaitu case folding untuk mengubah seluruh huruf menjadi lowercase dan cleaning karakter untuk menghapus karakter yang tidak diperlukan, menata tanda baca, serta merapikan spasi. Angka dan tanda seru maupun tanda tanya tetap dipertahankan dalam teks hasil cleaning.

In [ ]:
# ============================================
# STEP 4: TEXT PREPROCESSING DAN CLEANING
# ============================================

def clean_text(text):
    # Tahap 1: Case Folding
    text = str(text).lower()

    # Tahap 2: Cleaning Karakter
    # Pertahankan huruf, angka, spasi, ! dan ?
    text = re.sub(r"[^a-z0-9\s!?]", " ", text)

    # Pisahkan tanda baca dari kata
    text = re.sub(r"([!?])", r" \1 ", text)

    # Rapikan spasi berlebih
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Terapkan cleaning
df["title"] = df["title"].apply(clean_text)

print("=== HASIL PREPROCESSING ===")
print("Jumlah data:", len(df))

display(
    df[["title", "label", "label_score"]].head(10)
)

print("\n[OK] Text preprocessing selesai.")

## Step 5: Pemeriksaan Duplikat Setelah Cleaning

Memeriksa kembali judul yang menjadi identik setelah proses preprocessing. Duplikat tambahan dihapus dengan mempertahankan kemunculan pertama agar tidak terdapat judul yang sama dalam dataset akhir.

In [ ]:
# ============================================
# STEP 5: PEMERIKSAAN DUPLIKAT SETELAH CLEANING
# ============================================

jumlah_sebelum = len(df)

# Identifikasi duplikat setelah cleaning
audit_duplikat_clean = df[
    df.duplicated(
        subset="title",
        keep=False
    )
].copy()

print("=== DUPLIKAT SETELAH CLEANING ===")

display(
    audit_duplikat_clean[
        ["excel_row", "title", "label"]
    ].sort_values("title")
)

# Pastikan tidak ada konflik label baru
konflik_baru = (
    df.groupby("title")["label"].nunique() > 1
).sum()

assert konflik_baru == 0, (
    "Terdapat judul identik dengan label berbeda "
    "setelah cleaning. Periksa sebelum menghapus."
)

# Hapus duplikat tambahan
df = df.drop_duplicates(
    subset="title",
    keep="first"
).reset_index(drop=True)

jumlah_dihapus = jumlah_sebelum - len(df)

print("\n=== HASIL PENYARINGAN ===")
print(f"Data sebelum          : {jumlah_sebelum}")
print(f"Duplikat dihapus      : {jumlah_dihapus}")
print(f"Data setelah          : {len(df)}")

assert jumlah_dihapus == 2
assert len(df) == 8596

print("\n[OK] Duplikat setelah cleaning berhasil dihapus.")

## Step 6: Validasi Akhir Dataset

Memastikan dataset hasil preprocessing memiliki jumlah data yang sesuai, tidak mengandung judul kosong atau duplikat, serta memiliki label klasifikasi yang valid dan konsisten.

In [ ]:
# ============================================
# STEP 6: VALIDASI AKHIR DATASET
# ============================================

print("=== VALIDASI AKHIR DATASET ===")

# 1. Validasi jumlah data
assert len(df) == 8596, (
    f"Jumlah data tidak sesuai: {len(df)}"
)

# 2. Validasi judul tidak kosong
assert df["title"].notna().all()
assert df["title"].str.strip().ne("").all()

# 3. Validasi tidak ada duplikat
assert not df["title"].duplicated().any()

# 4. Validasi label
assert df["label"].isin([
    "clickbait", "non-clickbait"
]).all()

# 5. Validasi encoding label
label_expected = df["label"].map({
    "non-clickbait": 0,
    "clickbait": 1
}).astype(int)

assert df["label_score"].eq(label_expected).all()

print("Jumlah data:", len(df))
print("Judul kosong:", df["title"].eq("").sum())
print("Judul duplikat:", df["title"].duplicated().sum())

print("\nDistribusi label:")
print(df["label"].value_counts())

print("\n[OK] Seluruh validasi berhasil.")

## Step 7: Menyimpan Dataset Hasil Preprocessing

Menyimpan dataset akhir dalam format CSV untuk digunakan pada tahap ekstraksi fitur dan pelatihan model. File berisi kolom title, label, dan label_score.

In [ ]:
# ============================================
# STEP 7: MENYIMPAN DATASET HASIL PREPROCESSING
# ============================================

# Ambil kolom final
df = df[
    ["title", "label", "label_score"]
].copy().reset_index(drop=True)

# Lokasi file CSV
CLEAN_DATA_PATH = os.path.join(
    DATA_DIR,
    "dataset_preprocessed.csv"
)

# Simpan hasil preprocessing
df.to_csv(
    CLEAN_DATA_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Verifikasi file tersimpan
assert os.path.exists(CLEAN_DATA_PATH)
assert len(df) == 8596

print("=== PREPROCESSING SELESAI ===")
print("Jumlah data akhir:", len(df))
print("Lokasi CSV:", CLEAN_DATA_PATH)

print("\nPreview dataset akhir:")
display(df.head(10))

print("\n[OK] Dataset siap digunakan untuk training.")

# Bagian 2 - *Training* dan Evaluasi Model

Melakukan pembagian dataset, ekstraksi fitur TF-IDF, pelatihan model Logistic Regression, Naive Bayes, dan Multilayer Perceptron (MLP), serta evaluasi performa menggunakan metrik klasifikasi dan visualisasi.

## Step 1: Membaca Dataset Hasil Preprocessing

Membaca dataset yang telah melalui proses preprocessing dan validasi pada Bagian 1. Dataset berisi judul berita dan label klasifikasi yang selanjutnya digunakan dalam proses pembagian data, ekstraksi fitur, dan pelatihan model.


In [ ]:
# ============================================
# STEP 1: MEMBACA DATASET HASIL PREPROCESSING
# ============================================

# Membaca CSV hasil Bagian 1
dataset = pd.read_csv(CLEAN_DATA_PATH)

# Validasi struktur dataset
kolom_wajib = ["title", "label", "label_score"]

assert all(
    kolom in dataset.columns for kolom in kolom_wajib
), "Kolom dataset tidak lengkap."

assert len(dataset) == 8596, (
    f"Jumlah data tidak sesuai: {len(dataset)}"
)

assert dataset["title"].notna().all()
assert dataset["label_score"].isin([0, 1]).all()
assert not dataset["title"].duplicated().any()

# Ringkasan dataset
print("=== DATASET HASIL PREPROCESSING ===")
print("Jumlah data:", len(dataset))
print("Jumlah kolom:", len(dataset.columns))

print("\nDistribusi label:")
print(dataset["label"].value_counts())

print("\nPreview dataset:")
display(dataset.head(10))

print("\n[OK] Dataset siap digunakan untuk training.")

## Step 2: Pembagian Dataset dan Ekstraksi Fitur TF-IDF

Membagi dataset secara stratified menjadi 70% data training, 10% validation, dan 20% testing untuk mempertahankan proporsi kelas pada setiap subset. Selanjutnya, teks judul berita diubah menjadi representasi numerik menggunakan Term Frequency-Inverse Document Frequency (TF-IDF) dengan kombinasi unigram dan bigram.

Proses fitting TF-IDF hanya dilakukan pada data training untuk mencegah data leakage.


In [ ]:
# ============================================
# STEP 2: SPLIT DATA DAN EKSTRAKSI FITUR TF-IDF
# ============================================

SEED = 42

# ============================================
# 1. PEMBAGIAN DATASET (70/10/20)
# ============================================

# Tahap pertama: 80% train+val, 20% test
train_val, test = train_test_split(
    dataset,
    test_size=0.2,
    random_state=SEED,
    stratify=dataset["label_score"]
)

# Tahap kedua: 70% train, 10% validation
train, val = train_test_split(
    train_val,
    test_size=0.125,
    random_state=SEED,
    stratify=train_val["label_score"]
)

print("=== PEMBAGIAN DATASET ===")
print(f"Training   : {len(train)} data")
print(f"Validation : {len(val)} data")
print(f"Testing    : {len(test)} data")
print(f"Total      : {len(dataset)} data")

assert len(train) + len(val) + len(test) == 8596

# ============================================
# 2. EKSTRAKSI FITUR TF-IDF
# ============================================

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

# Fit hanya pada training set
X_train = tfidf.fit_transform(
    train["title"].astype(str)
)

# Transform validation dan test set
X_val = tfidf.transform(
    val["title"].astype(str)
)

X_test = tfidf.transform(
    test["title"].astype(str)
)

# Label numerik
y_train = train["label_score"].to_numpy()
y_val = val["label_score"].to_numpy()
y_test = test["label_score"].to_numpy()

print("\n=== HASIL EKSTRAKSI TF-IDF ===")
print("Jumlah fitur:", len(tfidf.get_feature_names_out()))
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

# ============================================
# 3. MENYIMPAN HASIL TRANSFORMASI
# ============================================

# Simpan TF-IDF vectorizer
joblib.dump(
    tfidf,
    f"{OUTPUT_DIR}/tfidf_vectorizer.pkl"
)

# Simpan matriks sparse
sp.save_npz(
    f"{OUTPUT_DIR}/X_train.npz", X_train
)

sp.save_npz(
    f"{OUTPUT_DIR}/X_val.npz", X_val
)

sp.save_npz(
    f"{OUTPUT_DIR}/X_test.npz", X_test
)

# Simpan label
np.save(f"{OUTPUT_DIR}/y_train.npy", y_train)
np.save(f"{OUTPUT_DIR}/y_val.npy", y_val)
np.save(f"{OUTPUT_DIR}/y_test.npy", y_test)

print("\n[OK] Split data dan TF-IDF selesai.")
print("[OK] Vectorizer dan hasil transformasi tersimpan.")

## Step 3: Pelatihan Model Logistic Regression, Naive Bayes, dan MLP

Melatih tiga model klasifikasi menggunakan fitur TF-IDF, yaitu Logistic Regression dan Naive Bayes sebagai baseline serta Multilayer Perceptron (MLP) sebagai model utama.

Model MLP menggunakan satu hidden layer dengan 64 neuron dan aktivasi ReLU, regularisasi L2, Dropout, serta output sigmoid. Pelatihan menggunakan optimizer Adam dan Binary Cross-Entropy Loss, dengan EarlyStopping dan ReduceLROnPlateau untuk mengendalikan proses pembelajaran.


In [ ]:
# ============================================
# STEP 3: PELATIHAN MODEL LR, NB, DAN MLP
# ============================================

# Reproduktivitas eksperimen
keras.utils.set_random_seed(SEED)

# ============================================
# 1. LOGISTIC REGRESSION
# ============================================

print("=== TRAINING LOGISTIC REGRESSION ===")

lr_model = LogisticRegression(
    max_iter=2000,
    random_state=SEED
)

lr_model.fit(X_train, y_train)

joblib.dump(
    lr_model,
    f"{OUTPUT_DIR}/logistic_regression.pkl"
)

print("[OK] Logistic Regression selesai dilatih.")


# ============================================
# 2. NAIVE BAYES
# ============================================

print("\n=== TRAINING NAIVE BAYES ===")

nb_model = MultinomialNB(alpha=1.0)

nb_model.fit(X_train, y_train)

joblib.dump(
    nb_model,
    f"{OUTPUT_DIR}/naive_bayes.pkl"
)

print("[OK] Naive Bayes selesai dilatih.")


# ============================================
# 3. ARSITEKTUR MODEL MLP
# ============================================

def build_mlp(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        layers.Dense(
            64,
            activation="relu",
            kernel_regularizer=keras.regularizers.l2(1e-4)
        ),

        layers.Dropout(0.4),

        layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(3e-4),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall")
        ]
    )

    return model


# ============================================
# 4. PEMBOBOTAN KELAS
# ============================================

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight = dict(zip(
    classes.astype(int),
    weights
))

print("\n=== CLASS WEIGHT ===")
print(class_weight)


# ============================================
# 5. CALLBACK TRAINING
# ============================================

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )
]


# ============================================
# 6. TRAINING MLP
# ============================================

mlp_model = build_mlp(X_train.shape[1])

print("\n=== ARSITEKTUR MODEL MLP ===")
mlp_model.summary()

print("\n=== TRAINING MLP ===")

history_mlp = mlp_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)


# ============================================
# 7. MENYIMPAN MODEL DAN HISTORY
# ============================================

# Simpan model MLP
mlp_model.save(
    f"{OUTPUT_DIR}/mlp_model.keras"
)

# Simpan riwayat training
history_df = pd.DataFrame(
    history_mlp.history
)

history_df.to_csv(
    f"{OUTPUT_DIR}/history_mlp.csv",
    index=False
)

print("\n=== TRAINING SELESAI ===")
print("[OK] Logistic Regression tersimpan.")
print("[OK] Naive Bayes tersimpan.")
print("[OK] MLP tersimpan.")
print("[OK] Riwayat training MLP tersimpan.")

## Step 4: Evaluasi dan Visualisasi Model

Mengevaluasi performa Logistic Regression, Naive Bayes, dan MLP menggunakan test set berdasarkan metrik accuracy, precision, recall, dan F1-score.

Selanjutnya, dilakukan visualisasi kurva training dan validation loss, kurva training dan validation accuracy, serta confusion matrix pada model MLP untuk menganalisis proses pembelajaran dan kesalahan klasifikasi.

In [ ]:
# ============================================
# STEP 4: EVALUASI DAN VISUALISASI MODEL
# ============================================

# ============================================
# 1. PREDIKSI TEST SET
# ============================================

print("=== EVALUASI MODEL PADA TEST SET ===")

# Logistic Regression
y_pred_lr = lr_model.predict(X_test)

# Naive Bayes
y_pred_nb = nb_model.predict(X_test)

# MLP
y_prob_mlp = mlp_model.predict(
    X_test,
    verbose=0
).ravel()

y_pred_mlp = (y_prob_mlp >= 0.5).astype(int)


# ============================================
# 2. TABEL PERBANDINGAN MODEL
# ============================================

def hitung_metrik(nama_model, prediksi):
    return {
        "Model": nama_model,

        "Accuracy (%)":
            accuracy_score(y_test, prediksi) * 100,

        "Precision (%)":
            precision_score(
                y_test, prediksi,
                pos_label=1, zero_division=0
            ) * 100,

        "Recall (%)":
            recall_score(
                y_test, prediksi,
                pos_label=1, zero_division=0
            ) * 100,

        "F1-Score (%)":
            f1_score(
                y_test, prediksi,
                pos_label=1, zero_division=0
            ) * 100,

        "F1 Macro (%)":
            f1_score(
                y_test, prediksi,
                average="macro",
                zero_division=0
            ) * 100
    }


tabel_hasil = pd.DataFrame([
    hitung_metrik("Logistic Regression", y_pred_lr),
    hitung_metrik("Naive Bayes", y_pred_nb),
    hitung_metrik("MLP", y_pred_mlp)
])

print("\n=== TABEL PERBANDINGAN MODEL ===")
print("Precision, Recall, dan F1-Score: kelas Clickbait")

display(tabel_hasil.round(2))

tabel_hasil.round(4).to_csv(
    f"{OUTPUT_DIR}/tabel_perbandingan.csv",
    index=False
)


# ============================================
# 3. INFORMASI TRAINING MLP
# ============================================

riwayat = pd.read_csv(
    f"{OUTPUT_DIR}/history_mlp.csv"
)

epochs = np.arange(1, len(riwayat) + 1)

best_epoch = int(
    riwayat["val_loss"].idxmin() + 1
)

print("\n=== INFORMASI TRAINING MLP ===")
print(f"Jumlah epoch    : {len(riwayat)}")
print(f"Epoch terbaik   : {best_epoch}")
print(
    f"Validation loss : "
    f"{riwayat['val_loss'].min():.4f}"
)


# ============================================
# 4. KURVA TRAINING & VALIDATION LOSS
# ============================================

plt.figure(figsize=(9, 5))

plt.plot(
    epochs,
    riwayat["loss"],
    label="Training Loss",
    linewidth=2
)

plt.plot(
    epochs,
    riwayat["val_loss"],
    label="Validation Loss",
    linewidth=2
)

plt.axvline(
    best_epoch,
    color="gray",
    linestyle="--",
    alpha=0.8,
    label=f"Best Epoch: {best_epoch}"
)

plt.title("MLP - Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (Binary Cross-Entropy + L2)")

plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/kurva_loss_mlp.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()


# ============================================
# 5. KURVA TRAINING & VALIDATION ACCURACY
# ============================================

plt.figure(figsize=(9, 5))

plt.plot(
    epochs,
    riwayat["accuracy"],
    label="Training Accuracy",
    linewidth=2
)

plt.plot(
    epochs,
    riwayat["val_accuracy"],
    label="Validation Accuracy",
    linewidth=2
)

plt.axvline(
    best_epoch,
    color="gray",
    linestyle="--",
    alpha=0.8,
    label=f"Best Epoch: {best_epoch}"
)

plt.title("MLP - Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)

plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/kurva_accuracy_mlp.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()


# ============================================
# 6. CONFUSION MATRIX MLP
# ============================================

labels = [0, 1]

nama_kelas = [
    "Non-clickbait",
    "Clickbait"
]

cm = confusion_matrix(
    y_test,
    y_pred_mlp,
    labels=labels
)

fig, ax = plt.subplots(figsize=(7, 5.5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=nama_kelas
)

disp.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

ax.set_title("Confusion Matrix - MLP (Test Set)")

plt.tight_layout()

fig.savefig(
    f"{OUTPUT_DIR}/confusion_matrix_mlp.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

# Simpan angka confusion matrix
pd.DataFrame(
    cm,
    index=[
        "Aktual Non-clickbait",
        "Aktual Clickbait"
    ],
    columns=[
        "Prediksi Non-clickbait",
        "Prediksi Clickbait"
    ]
).to_csv(
    f"{OUTPUT_DIR}/confusion_matrix_mlp.csv"
)


# ============================================
# 7. CLASSIFICATION REPORT MLP
# ============================================

print("\n=== CLASSIFICATION REPORT - MLP ===")

print(
    classification_report(
        y_test,
        y_pred_mlp,
        labels=labels,
        target_names=nama_kelas,
        digits=4,
        zero_division=0
    )
)

report = classification_report(
    y_test,
    y_pred_mlp,
    labels=labels,
    target_names=nama_kelas,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report).transpose().to_csv(
    f"{OUTPUT_DIR}/classification_report_mlp.csv"
)


# ============================================
# 8. RINGKASAN ARSITEKTUR MLP
# ============================================

print("\n=== RINGKASAN MODEL MLP ===")

mlp_model.summary()

print(
    "\nTotal parameter:",
    mlp_model.count_params()
)

print("\n[OK] Evaluasi dan visualisasi selesai.")
print(f"[OK] Hasil tersimpan di: {OUTPUT_DIR}")

# Bagian Akhir - Penyimpanan Hasil

Melakukan pemeriksaan kelengkapan file hasil preprocessing, pelatihan, dan evaluasi model. Seluruh file kemudian dikemas ke dalam satu arsip ZIP agar dapat diunduh dan digunakan kembali tanpa memerlukan Google Drive.

## Step 1: Memeriksa dan Menyiapkan File Hasil

Memvalidasi keberadaan dataset hasil preprocessing, model terlatih, ekstraksi fitur, riwayat pelatihan, hasil evaluasi, dan visualisasi. Seluruh file yang diperlukan disalin ke direktori hasil akhir untuk memastikan kelengkapan output proyek.

In [ ]:
# ============================================
# STEP 1: MENYIAPKAN FILE HASIL AKHIR
# ============================================

# Direktori ekspor dan subfolder
EXPORT_DIR = os.path.join(
    BASE_DIR, "kelompok2_hasil_uts_clickbait"
)

EXPORT_DATA_DIR = os.path.join(EXPORT_DIR, "data")
EXPORT_MODEL_DIR = os.path.join(EXPORT_DIR, "models")

# Daftar file output training dan evaluasi
required_files = [
    "tfidf_vectorizer.pkl",
    "X_train.npz",
    "X_val.npz",
    "X_test.npz",
    "y_train.npy",
    "y_val.npy",
    "y_test.npy",
    "logistic_regression.pkl",
    "naive_bayes.pkl",
    "mlp_model.keras",
    "history_mlp.csv",
    "tabel_perbandingan.csv",
    "classification_report_mlp.csv",
    "confusion_matrix_mlp.csv",
    "kurva_loss_mlp.png",
    "kurva_accuracy_mlp.png",
    "confusion_matrix_mlp.png"
]

# ============================================
# 1. VALIDASI FILE
# ============================================

# Validasi dataset utama dan hasil preprocessing
assert os.path.isfile(DATA_PATH), (
    "Dataset utama tidak ditemukan."
)

assert os.path.isfile(CLEAN_DATA_PATH), (
    "Dataset preprocessing tidak ditemukan."
)

# Validasi file model dan evaluasi
missing_files = [
    name for name in required_files
    if not os.path.isfile(
        os.path.join(OUTPUT_DIR, name)
    )
]

if missing_files:
    raise FileNotFoundError(
        "File hasil belum lengkap:\n"
        + "\n".join(missing_files)
    )

# Validasi jumlah dataset
data_final = pd.read_csv(CLEAN_DATA_PATH)

assert len(data_final) == 8596, (
    "Jumlah dataset akhir tidak sesuai."
)

print("[OK] Seluruh file berhasil divalidasi.")


# ============================================
# 2. MEMBUAT STRUKTUR FOLDER
# ============================================

# Bersihkan hasil ekspor sebelumnya
if os.path.exists(EXPORT_DIR):
    shutil.rmtree(EXPORT_DIR)

os.makedirs(EXPORT_DATA_DIR, exist_ok=True)
os.makedirs(EXPORT_MODEL_DIR, exist_ok=True)


# ============================================
# 3. MENYALIN FILE DATASET
# ============================================

# Dataset utama
shutil.copy2(
    DATA_PATH,
    os.path.join(
        EXPORT_DATA_DIR, "main_dataset.xlsx"
    )
)

# Dataset hasil preprocessing
shutil.copy2(
    CLEAN_DATA_PATH,
    os.path.join(
        EXPORT_DATA_DIR, "dataset_preprocessed.csv"
    )
)


# ============================================
# 4. MENYALIN MODEL DAN HASIL EVALUASI
# ============================================

for name in required_files:
    shutil.copy2(
        os.path.join(OUTPUT_DIR, name),
        os.path.join(EXPORT_MODEL_DIR, name)
    )


# ============================================
# 5. MEMBUAT RINGKASAN EKSPERIMEN
# ============================================

riwayat = pd.read_csv(
    os.path.join(OUTPUT_DIR, "history_mlp.csv")
)

hasil_evaluasi = pd.read_csv(
    os.path.join(OUTPUT_DIR, "tabel_perbandingan.csv")
)

epoch_terbaik = int(
    riwayat["val_loss"].idxmin() + 1
)

ringkasan = (
    "HASIL EKSPERIMEN UTS - DEEP LEARNING\n"
    "====================================\n\n"
    "Kelompok: 2\n"
    "Topik: Klasifikasi Judul Berita Clickbait\n"
    "Dataset: CLICK-ID (All-Agree)\n"
    f"Jumlah data akhir: {len(data_final)}\n"
    f"Training: {len(y_train)}\n"
    f"Validation: {len(y_val)}\n"
    f"Testing: {len(y_test)}\n"
    f"Random seed: {SEED}\n\n"
    "Model: Logistic Regression, Naive Bayes, MLP\n"
    "Ekstraksi fitur: TF-IDF\n\n"
    f"Total epoch MLP: {len(riwayat)}\n"
    f"Epoch terbaik MLP: {epoch_terbaik}\n"
    f"Validation loss terbaik: "
    f"{riwayat['val_loss'].min():.4f}\n\n"
    "HASIL EVALUASI TEST SET\n"
    "=======================\n"
    + hasil_evaluasi.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
    + "\n\n"
    "Label 0 = Non-clickbait\n"
    "Label 1 = Clickbait\n"
    "Precision, Recall, dan F1-Score pada tabel "
    "mengacu pada kelas Clickbait.\n"
)

summary_path = os.path.join(
    EXPORT_MODEL_DIR,
    "ringkasan_eksperimen.txt"
)

with open(summary_path, "w", encoding="utf-8") as f:
    f.write(ringkasan)


# ============================================
# 6. MENAMPILKAN STRUKTUR HASIL
# ============================================

print("\n=== FILE HASIL AKHIR ===")

for folder in [EXPORT_DATA_DIR, EXPORT_MODEL_DIR]:
    print(f"\n{os.path.basename(folder)}/")

    for name in sorted(os.listdir(folder)):
        path = os.path.join(folder, name)
        ukuran = os.path.getsize(path) / 1024

        print(f"  {name} ({ukuran:.1f} KB)")

print("\n[OK] Seluruh file siap dikemas.")
print("Lokasi:", EXPORT_DIR)

## Step 2: Mengemas dan Mengunduh Seluruh Hasil

Mengompres seluruh dataset hasil preprocessing, model terlatih, riwayat pelatihan, metrik evaluasi, dan visualisasi menjadi satu file ZIP. File tersebut kemudian dapat diunduh langsung dari Google Colab.

In [ ]:
# ============================================
# STEP 2: MENGEMAS DAN MENGUNDUH HASIL
# ============================================

# Nama dan lokasi file ZIP
ZIP_NAME = "kelompok2_hasil_uts_clickbait.zip"

ZIP_PATH = os.path.join(
    BASE_DIR, ZIP_NAME
)

# Validasi folder ekspor
for folder in [EXPORT_DATA_DIR, EXPORT_MODEL_DIR]:
    if not os.path.isdir(folder):
        raise FileNotFoundError(
            f"Folder tidak ditemukan: {folder}"
        )

    if not os.listdir(folder):
        raise ValueError(
            f"Folder masih kosong: {folder}"
        )


# ============================================
# 1. MEMBUAT FILE ZIP
# ============================================

print("=== MENGEMAS HASIL EKSPERIMEN ===")

shutil.make_archive(
    base_name=os.path.splitext(ZIP_PATH)[0],
    format="zip",
    root_dir=EXPORT_DIR
)

# Validasi file ZIP
assert os.path.isfile(ZIP_PATH)

ukuran_zip = (
    os.path.getsize(ZIP_PATH) / (1024 * 1024)
)

print("\n[OK] File ZIP berhasil dibuat.")
print("Nama file :", ZIP_NAME)
print("Ukuran    :", f"{ukuran_zip:.2f} MB")
print("Lokasi    :", ZIP_PATH)

print("\nStruktur ZIP:")
print("  data/")
print("  models/")


# ============================================
# 2. MENGUNDUH FILE ZIP
# ============================================

print("\nMemulai download file hasil...")

files.download(ZIP_PATH)